In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
from collections import Counter
from itertools import combinations
import seaborn as sns
warnings.filterwarnings("ignore")

In [2]:
# Load data
df = pd.read_csv("train.csv")

In [10]:
df["capturedAt"] = pd.to_datetime(df["capturedAt"])

In [3]:
# shopIdList
shopId_list = df["shopId"].unique().tolist()

In [4]:
len(shopId_list)

219

## Offifical Shop

In [5]:
# is_official_shop
df_shop_official_status = (
    df[["shopId", "is_official_shop"]]
    .copy()
)
df_shop_official_status = df_shop_official_status.drop_duplicates(keep="first")

In [6]:
len(shopId_list) == len(df_shop_official_status)

True

In [7]:
df_shop_official_status["is_official_shop"].value_counts()

is_official_shop
f    205
t     14
Name: count, dtype: int64

## Verified Status

In [8]:
# is_verified_status
df_shop_verified_status = (
    df[["shopId", "is_verified"]]
    .copy()
)
df_shop_verified_status = df_shop_verified_status.drop_duplicates(keep="first")

In [9]:
len(shopId_list) == len(df_shop_verified_status)

False

In [11]:
df_shop_verified_status = (
    df[["shopId", "is_verified", "capturedAt"]]
    .copy()
)
df_shop_verified_status = df_shop_verified_status.drop_duplicates(keep="first")
df_shop_verified_status["capturedAt"] = pd.to_datetime(
    df_shop_verified_status["capturedAt"]
).dt.date

dates = pd.date_range(
    start=df_shop_verified_status['capturedAt'].min(),
    end=df_shop_verified_status['capturedAt'].max()
).date
df_shop_verified_daily = (
    df_shop_verified_status
    .pivot_table(
        index="shopId",
        columns="capturedAt",
        values="is_verified",
        aggfunc="first"
    )
    .reindex(columns=dates)
    .reset_index()
)
df_shop_verified_daily.head(5)

capturedAt,shopId,2025-01-01,2025-01-02,2025-01-03,2025-01-04,2025-01-05,2025-01-06,2025-01-07,2025-01-08,2025-01-09,...,2025-03-13,2025-03-14,2025-03-15,2025-03-16,2025-03-17,2025-03-18,2025-03-19,2025-03-20,2025-03-21,2025-03-22
0,1000692,NaN,NaN,f,NaN,NaN,f,NaN,f,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1001138,NaN,NaN,NaN,NaN,NaN,f,NaN,f,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1001950,NaN,NaN,NaN,NaN,f,f,NaN,NaN,NaN,...,f,f,NaN,NaN,f,NaN,NaN,NaN,NaN,NaN
3,1003777,NaN,NaN,f,NaN,NaN,NaN,NaN,NaN,NaN,...,f,f,f,f,f,f,f,NaN,f,NaN
4,1004468,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,f,f,f,f,f,f,f,f,f,f


In [12]:
def analyze_shop_status_changes(df_shop_verified_daily):
    changes = []

    # All date columns except the shop identifier
    date_cols = [
        c
        for c in df_shop_verified_daily.columns
        if c != "shopId"
    ]

    # Detect status changes for every shop
    for _, row in df_shop_verified_daily.iterrows():
        shop_id = row["shopId"]

        prev_value = None
        change_list = []

        for date in date_cols:
            value = row[date]

            # Ignore missing values
            if pd.isna(value):
                continue

            # Detect status transition
            if prev_value is not None and value != prev_value:
                direction = f"{prev_value}->{value}"
                change_list.append((date, direction))

            prev_value = value

        if change_list:
            changes.append((shop_id, change_list))

    # Shops with more than one change
    multi_changes = [
        (shop_id, change_list)
        for shop_id, change_list in changes
        if len(change_list) > 1
    ]

    # Shops with at least one False -> True change
    f_to_t_changes = [
        (
            shop_id,
            [
                (date, direction)
                for date, direction in change_list
                if direction == "f->t"
            ],
        )
        for shop_id, change_list in changes
        if any(direction == "f->t" for _, direction in change_list)
    ]

    # Shops with at least one True -> False change
    t_to_f_changes = [
        (
            shop_id,
            [
                (date, direction)
                for date, direction in change_list
                if direction == "t->f"
            ],
        )
        for shop_id, change_list in changes
        if any(direction == "t->f" for _, direction in change_list)
    ]

    # Count change frequency by (date, direction)
    date_change_counter = Counter()

    for _, change_list in changes:
        date_change_counter.update(change_list)

    # Summary
    print(f"Total shops with changes: {len(changes)}")
    print(f"Shops with multiple changes: {len(multi_changes)}")
    print(f"Shops with at least one False -> True change: {len(f_to_t_changes)}")
    print(f"Shops with at least one True -> False change: {len(t_to_f_changes)}")

    return (
        changes,
        multi_changes,
        f_to_t_changes,
        t_to_f_changes,
        date_change_counter,
    )

In [13]:
(
    changes,
    multi_changes,
    f_to_t_changes,
    t_to_f_changes,
    date_change_counter,
) = analyze_shop_status_changes(df_shop_verified_daily)

Total shops with changes: 22
Shops with multiple changes: 9
Shops with at least one False -> True change: 15
Shops with at least one True -> False change: 16


In [14]:
date_change_counter

Counter({(datetime.date(2025, 2, 2), 'f->t'): 6,
         (datetime.date(2025, 2, 25), 't->f'): 2,
         (datetime.date(2025, 2, 18), 't->f'): 2,
         (datetime.date(2025, 2, 2), 't->f'): 2,
         (datetime.date(2025, 2, 7), 'f->t'): 2,
         (datetime.date(2025, 3, 4), 'f->t'): 1,
         (datetime.date(2025, 2, 13), 'f->t'): 1,
         (datetime.date(2025, 2, 6), 't->f'): 1,
         (datetime.date(2025, 1, 7), 't->f'): 1,
         (datetime.date(2025, 3, 11), 'f->t'): 1,
         (datetime.date(2025, 3, 17), 't->f'): 1,
         (datetime.date(2025, 2, 5), 'f->t'): 1,
         (datetime.date(2025, 2, 17), 't->f'): 1,
         (datetime.date(2025, 1, 10), 'f->t'): 1,
         (datetime.date(2025, 2, 11), 't->f'): 1,
         (datetime.date(2025, 3, 18), 't->f'): 1,
         (datetime.date(2025, 2, 13), 't->f'): 1,
         (datetime.date(2025, 2, 12), 't->f'): 1,
         (datetime.date(2025, 3, 4), 't->f'): 1,
         (datetime.date(2025, 3, 10), 'f->t'): 1,
        